In [1]:
import pandas as pd

In [4]:
df = pd.read_csv("dataset_energy.csv")
df.head(5)

,ID,Статистическое время,Средняя скорость ветра(m/s),Нормализованная активная мощность,Средняя температура окружающей среды(°C)
0,1,2023-03-11 0:00:00,6.62,0.37,15.52
1,2,2023-03-11 0:10:00,6.70,0.39,15.38
2,3,2023-03-11 0:20:00,7.14,0.44,15.43
3,4,2023-03-11 0:30:00,7.82,0.50,15.41
4,5,2023-03-11 0:40:00,7.62,0.49,15.42


In [7]:
# Приведение названий к удобному виду
df.columns = [
    "ID",
    "datetime",
    "wind_speed",
    "target_power",
    "temperature"
]

df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

print("Размер:", df.shape)
print("\nТипы данных:")
print(df.dtypes)

print("\nПропуски:")
missing = pd.DataFrame({
    "Количество": df.isna().sum(),
    "Процент": (df.isna().mean() * 100).round(2)
})
print(missing)

print("\nПолные дубликаты:", df.duplicated().sum())
print("Повторяющиеся даты:", df["datetime"].duplicated().sum())

print("\nОсновная статистика:")
print(df.describe(include="all"))

# Проверка временной сетки
df = df.sort_values("datetime")
df["time_gap"] = df["datetime"].diff()

print("\nРаспределение интервалов:")
print(df["time_gap"].value_counts().head(10))

print("\nРазрывы больше 10 минут:")
print(df.loc[df["time_gap"] > pd.Timedelta(minutes=10),
             ["datetime", "time_gap"]].head(20))

# Проверка физических ограничений
print("Отрицательная скорость ветра:",
      (df["wind_speed"] < 0).sum())

print("Мощность меньше 0:",
      (df["target_power"] < 0).sum())

print("Мощность больше 1:",
      (df["target_power"] > 1).sum())

Размер: (149499, 5)

Типы данных:
ID                       int64
datetime        datetime64[us]
wind_speed             float64
target_power           float64
temperature            float64
dtype: object

Пропуски:
              Количество  Процент
ID                     0      0.0
datetime               0      0.0
wind_speed             0      0.0
target_power           0      0.0
temperature            0      0.0

Полные дубликаты: 0
Повторяющиеся даты: 0

Основная статистика:
                  ID                    datetime     wind_speed  \
count  149499.000000                      149499  149499.000000   
mean    74750.000000  2024-08-21 18:03:19.711035       6.462796   
min         1.000000         2023-03-11 00:00:00       0.110000   
25%     37375.500000         2023-11-30 23:25:00       3.240000   
50%     74750.000000         2024-08-23 04:20:00       6.130000   
75%    112124.500000         2025-05-12 15:55:00       9.440000   
max    149499.000000         2026-01-31 23:50:00

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

try:
    from catboost import CatBoostRegressor
except ImportError as exc:
    raise ImportError(
        "CatBoost не установлен. Выполните ячейку установки выше и перезапустите kernel."
    ) from exc

In [ ]:
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42

# ------------------------- НАСТРОЙКИ ПОЛЬЗОВАТЕЛЯ -------------------------
DATA_PATH = Path("wind_farm_data.csv")   # CSV или XLSX
USE_DEMO_DATA = False                    # True — запустить на синтетическом примере
CSV_SEPARATOR = None                     # None = определить автоматически
CSV_ENCODING = "utf-8"                  # при необходимости: "utf-8-sig" или "cp1251"

INSTALLED_CAPACITY_MW = None             # например, 50.0; None = прогноз только в долях 0...1
MIN_HOURLY_COVERAGE = 0.67               # минимум исходных точек внутри часа
INTERPOLATE_WEATHER_LIMIT_HOURS = 2      # только короткие разрывы погоды
DROP_STATISTICAL_ANOMALIES = False       # обычно False: остановки ВЭС могут быть реальными
RUN_WALK_FORWARD = False                 # True = дополнительная, более долгая кросс-валидация

ARTIFACT_DIR = Path("model_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
MANUAL_COLUMNS = {
    "datetime": None,       # например: "Статистическое время"
    "wind_speed": None,     # например: "Средняя скорость ветра(m/s)"
    "target_power": None,   # например: "Нормализованная активная мощность"
    "temperature": None,    # например: "Средняя температура окружающей среды(°C)"
}

COLUMN_ALIASES = {
    "datetime": [
        "статистическое время", "время", "дата", "datetime", "timestamp", "date_time", "time"
    ],
    "wind_speed": [
        "средняя скорость ветра", "скорость ветра", "wind speed", "wind_speed", "windspeed"
    ],
    "target_power": [
        "нормализованная активная мощность", "активная мощность", "normalized active power",
        "normalized_power", "target_power", "power"
    ],
    "temperature": [
        "средняя температура окружающей среды", "температура", "ambient temperature",
        "temperature", "temperature_2m", "temp"
    ],
}


def normalize_name(value):
    return " ".join(str(value).strip().lower().replace("_", " ").split())


def find_column(columns, aliases):
    normalized = {col: normalize_name(col) for col in columns}
    aliases = [normalize_name(x) for x in aliases]

    # Сначала точное совпадение, затем совпадение по подстроке.
    for col, name in normalized.items():
        if name in aliases:
            return col
    for alias in aliases:
        for col, name in normalized.items():
            if alias in name:
                return col
    return None


def load_table(path, separator=None, encoding="utf-8"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Файл не найден: {path.resolve()}\n"
            "Поместите CSV/XLSX рядом с ноутбуком или измените DATA_PATH."
        )

    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix in {".csv", ".txt"}:
        kwargs = {"encoding": encoding}
        if separator is None:
            kwargs.update({"sep": None, "engine": "python"})
        else:
            kwargs["sep"] = separator
        return pd.read_csv(path, **kwargs)
    raise ValueError("Поддерживаются форматы CSV, TXT, XLSX и XLS.")


def generate_demo_data(days=540, seed=42):
    '''Синтетические 10-минутные данные только для проверки выполнения ноутбука.'''
    rng = np.random.default_rng(seed)
    dt = pd.date_range("2024-01-01", periods=days * 24 * 6, freq="10min")
    seasonal = 1.3 * np.sin(2 * np.pi * dt.dayofyear.to_numpy() / 365.25)
    intraday = 0.5 * np.sin(2 * np.pi * dt.hour.to_numpy() / 24)
    wind = np.clip(7 + seasonal + intraday + rng.normal(0, 1.8, len(dt)), 0, 25)
    temp = 9 + 14 * np.sin(2 * np.pi * (dt.dayofyear.to_numpy() - 170) / 365.25)
    temp += rng.normal(0, 2.2, len(dt))

    # Упрощённая S-образная кривая мощности с шумом.
    power = 1 / (1 + np.exp(-(wind - 7.5) / 1.5))
    power = np.clip(power + rng.normal(0, 0.035, len(dt)), 0, 1)

    frame = pd.DataFrame({
        "ID": np.arange(1, len(dt) + 1),
        "Статистическое время": dt,
        "Средняя скорость ветра(m/s)": wind,
        "Нормализованная активная мощность": power,
        "Средняя температура окружающей среды(°C)": temp,
    })

    # Несколько дефектов для демонстрации аудита.
    missing_idx = rng.choice(len(frame), size=max(10, len(frame) // 300), replace=False)
    frame.loc[missing_idx[: len(missing_idx) // 2], "Средняя скорость ветра(m/s)"] = np.nan
    frame.loc[missing_idx[len(missing_idx) // 2 :], "Нормализованная активная мощность"] = np.nan
    return frame


raw = generate_demo_data() if USE_DEMO_DATA else load_table(
    DATA_PATH, separator=CSV_SEPARATOR, encoding=CSV_ENCODING
)

# Удаляем технический индекс, часто появляющийся после сохранения pandas.
unnamed = [c for c in raw.columns if normalize_name(c).startswith("unnamed")]
raw = raw.drop(columns=unnamed, errors="ignore")

resolved = {}
for canonical, aliases in COLUMN_ALIASES.items():
    resolved[canonical] = MANUAL_COLUMNS[canonical] or find_column(raw.columns, aliases)

missing_columns = [name for name, source in resolved.items() if source is None]
if missing_columns:
    raise ValueError(
        "Не удалось распознать поля: " + ", ".join(missing_columns) +
        ". Заполните MANUAL_COLUMNS. Доступные столбцы: " + str(list(raw.columns))
    )

df = raw.rename(columns={source: canonical for canonical, source in resolved.items()})[
    ["datetime", "wind_speed", "target_power", "temperature"]
].copy()

df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
for col in ["wind_speed", "target_power", "temperature"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Исходный размер: {raw.shape[0]:,} строк × {raw.shape[1]} столбцов")
print("Распознанные поля:")
display(pd.Series(resolved, name="исходный столбец").to_frame())
display(df.head())
